# 02 — Explore Library

Playlists, genres, tracks, artists — understand what's in the collection.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)

## 1. Playlist Overview

In [ ]:
# Tracks per playlist
playlist_tracks = conn.execute("""
    SELECT p.playlist_name, COUNT(*) as n_tracks
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    GROUP BY p.playlist_name
    ORDER BY n_tracks DESC
""").pl()

print(f"{len(playlist_tracks)} playlists, {playlist_tracks['n_tracks'].sum():,} total tracks")

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(
    playlist_tracks["playlist_name"].to_list()[::-1],
    playlist_tracks["n_tracks"].to_list()[::-1],
)
ax.set_xlabel("Track count")
ax.set_title("Tracks per Playlist")
plt.tight_layout()
plt.show()

## 2. Playlist Genre Profiles

In [ ]:
# Gen_4 distribution per playlist
playlist_genre = conn.execute("""
    SELECT p.playlist_name, tg.gen_4, COUNT(*) as n
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    LEFT JOIN track_genre tg ON pt.track_id = tg.track_id
    WHERE tg.gen_4 IS NOT NULL
    GROUP BY p.playlist_name, tg.gen_4
    ORDER BY p.playlist_name, n DESC
""").pl()

if not playlist_genre.is_empty():
    # Pivot for heatmap
    pivot = playlist_genre.pivot(on="gen_4", index="playlist_name", values="n").fill_null(0)
    genre_cols = [c for c in pivot.columns if c != "playlist_name"]

    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot) * 0.4)))
    data = pivot.select(genre_cols).to_numpy()
    # Normalize per playlist (row)
    row_sums = data.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    data_pct = data / row_sums

    sns.heatmap(
        data_pct,
        xticklabels=genre_cols,
        yticklabels=pivot["playlist_name"].to_list(),
        annot=True, fmt=".0%", cmap="YlOrRd", ax=ax,
    )
    ax.set_title("gen_4 Distribution per Playlist (row-normalized)")
    plt.tight_layout()
    plt.show()
else:
    print("No track_genre data available")

## 3. Artist Stats

In [ ]:
artist_stats = conn.execute("""
    SELECT
        a.artist_name,
        a.popularity,
        COUNT(DISTINCT ta.track_id) as n_tracks,
        COUNT(DISTINCT pt.playlist_id) as n_playlists
    FROM artists a
    JOIN track_artists ta ON a.artist_id = ta.artist_id
    LEFT JOIN playlist_tracks pt ON ta.track_id = pt.track_id
    GROUP BY a.artist_name, a.popularity
    ORDER BY n_tracks DESC
    LIMIT 30
""").pl()

print("Top 30 artists by track count:")
artist_stats

In [ ]:
# Artist popularity distribution
all_artists = conn.execute("SELECT popularity FROM artists WHERE popularity IS NOT NULL").pl()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_artists["popularity"].to_numpy(), bins=50, edgecolor="white", alpha=0.8)
ax.set_xlabel("Popularity")
ax.set_title(f"Artist Popularity Distribution (n={len(all_artists):,})")
plt.tight_layout()
plt.show()

### 3b. Artist Genre Tag Vocabulary

Raw Spotify genre tags — how they map to our curated `gen_4`/`gen_8` taxonomy.

In [ ]:
# Explode raw Spotify genre tags from artists table
# genres column stores stringified Python lists: "['latin', 'reggaeton']"
genre_tags = conn.execute("""
    WITH exploded AS (
        SELECT UNNEST(
            string_split(
                TRIM(BOTH '[]' FROM REPLACE(genres, '''', '')),
                ', '
            )
        ) AS tag
        FROM artists
        WHERE genres IS NOT NULL AND genres != '[]' AND genres != 'None'
    )
    SELECT tag, COUNT(*) AS n_artists
    FROM exploded
    WHERE TRIM(tag) != ''
    GROUP BY tag
    ORDER BY n_artists DESC
    LIMIT 25
""").pl()

n_tagged = conn.execute("""
    SELECT COUNT(*) FROM artists
    WHERE genres IS NOT NULL AND genres != '[]' AND genres != 'None'
""").fetchone()[0]
n_total = conn.execute("SELECT COUNT(*) FROM artists").fetchone()[0]
print(f"Artists with genre tags: {n_tagged:,} / {n_total:,}")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    genre_tags["tag"].to_list()[::-1],
    genre_tags["n_artists"].to_list()[::-1],
)
ax.set_xlabel("Artist count")
ax.set_title("Top 25 Raw Spotify Genre Tags")
plt.tight_layout()
plt.show()

# Compare to curated taxonomy coverage
if "gen_4" in corpus.columns:
    unmapped = corpus.filter(pl.col("gen_4").is_null())
    print(f"\nTracks without gen_4 mapping: {len(unmapped):,} / {len(corpus):,}")
    print("Raw tags give finer-grained signal than the curated gen_4/gen_8 taxonomy.")

### 3c. Artist Popularity vs Track Count

Which artists dominate the library? Color by cross-playlist presence.

In [ ]:
# Artist popularity vs track count, colored by playlist presence
artist_scatter = conn.execute("""
    SELECT
        a.artist_name,
        a.popularity,
        COUNT(DISTINCT ta.track_id) AS n_tracks,
        COUNT(DISTINCT pt.playlist_id) AS n_playlists
    FROM artists a
    JOIN track_artists ta ON a.artist_id = ta.artist_id
    LEFT JOIN playlist_tracks pt ON ta.track_id = pt.track_id
    WHERE a.popularity IS NOT NULL
    GROUP BY a.artist_name, a.popularity
""").pl()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    artist_scatter["popularity"].to_numpy(),
    artist_scatter["n_tracks"].to_numpy(),
    c=artist_scatter["n_playlists"].to_numpy(),
    cmap="viridis",
    alpha=0.6,
    s=25,
    edgecolors="white",
    linewidths=0.3,
)
plt.colorbar(scatter, ax=ax, label="Playlists containing artist")
ax.set_xlabel("Artist Popularity")
ax.set_ylabel("Track Count in Library")
ax.set_title(f"Artist Popularity vs Track Count (n={len(artist_scatter):,})")

# Annotate outliers: top 5 by track count
top_outliers = artist_scatter.sort("n_tracks", descending=True).head(5)
for row in top_outliers.iter_rows(named=True):
    ax.annotate(
        row["artist_name"],
        (row["popularity"], row["n_tracks"]),
        fontsize=8,
        alpha=0.8,
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.tight_layout()
plt.show()

## 4. Track Deep-Dive

In [ ]:
# Most/least popular tracks
if "popularity" in corpus.columns and "track_name" in corpus.columns:
    print("Top 10 most popular:")
    display(corpus.sort("popularity", descending=True).select(["track_name", "popularity", "decade", "gen_4"]).head(10))

    print("\nBottom 10 least popular:")
    display(corpus.sort("popularity").select(["track_name", "popularity", "decade", "gen_4"]).head(10))

In [ ]:
# Most-playlisted tracks
if "n_playlists" in corpus.columns:
    most_playlisted = corpus.sort("n_playlists", descending=True).head(15)
    print("Most-playlisted tracks:")
    most_playlisted.select(["track_name", "n_playlists", "popularity", "gen_4"])

In [ ]:
# Fave score distribution
if "fave_score" in corpus.columns:
    has_fave = corpus.filter(pl.col("fave_score") > 0)
    print(f"Tracks with fave_score > 0: {len(has_fave):,} / {len(corpus):,}")

    if not has_fave.is_empty():
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(has_fave["fave_score"].to_numpy(), bins=30, edgecolor="white", alpha=0.8)
        ax.set_xlabel("fave_score")
        ax.set_title("Fave Score Distribution (non-zero only)")
        plt.tight_layout()
        plt.show()

## 5. Temporal Patterns

In [ ]:
# How playlists differ in decade composition
decade_by_playlist = conn.execute("""
    SELECT p.playlist_name, t.decade, COUNT(*) as n
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    JOIN tracks t ON pt.track_id = t.track_id
    WHERE t.decade IS NOT NULL
    GROUP BY p.playlist_name, t.decade
""").pl()

if not decade_by_playlist.is_empty():
    pivot = decade_by_playlist.pivot(on="decade", index="playlist_name", values="n").fill_null(0)
    decade_cols = sorted([c for c in pivot.columns if c != "playlist_name"])

    fig, ax = plt.subplots(figsize=(12, max(6, len(pivot) * 0.4)))
    data = pivot.select(decade_cols).to_numpy()
    row_sums = data.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1

    sns.heatmap(
        data / row_sums,
        xticklabels=decade_cols,
        yticklabels=pivot["playlist_name"].to_list(),
        annot=True, fmt=".0%", cmap="Blues", ax=ax,
    )
    ax.set_title("Decade Distribution per Playlist")
    plt.tight_layout()
    plt.show()

### 5b. Decade x Popularity per Playlist

Which playlists skew older/newer, and how does that interact with track popularity?

In [ ]:
# Decade × popularity boxplot per playlist
decade_pop = conn.execute("""
    SELECT p.playlist_name, t.decade, t.popularity
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    JOIN tracks t ON pt.track_id = t.track_id
    WHERE t.decade IS NOT NULL AND t.popularity IS NOT NULL
""").pl()

if not decade_pop.is_empty():
    unique_playlists = decade_pop["playlist_name"].unique().sort().to_list()
    ncols_dp = 3
    nrows_dp = (len(unique_playlists) + ncols_dp - 1) // ncols_dp

    fig, axes = plt.subplots(nrows_dp, ncols_dp, figsize=(18, 5 * nrows_dp))
    axes = axes.flatten()

    decade_order = sorted(decade_pop["decade"].unique().to_list())

    for i, pname in enumerate(unique_playlists):
        pdata = decade_pop.filter(pl.col("playlist_name") == pname).to_pandas()
        sns.boxplot(
            data=pdata, x="decade", y="popularity",
            order=decade_order, ax=axes[i],
        )
        axes[i].set_title(pname, fontsize=10)
        axes[i].tick_params(axis="x", rotation=45, labelsize=8)
        axes[i].set_xlabel("")

    for j in range(len(unique_playlists), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Decade x Popularity per Playlist", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No decade/popularity data available")

## 6. Audio Feature Profiles (Radar Plots)

In [ ]:
RADAR_FEATURES = [
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
]

# Get per-playlist means
playlist_audio = conn.execute(f"""
    SELECT
        p.playlist_name,
        {', '.join(f'AVG(af.{f}) as {f}' for f in RADAR_FEATURES)}
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    JOIN audio_features af ON pt.track_id = af.track_id
    GROUP BY p.playlist_name
""").pl()

def radar_plot(df: pl.DataFrame, playlists: list[str], features: list[str]) -> None:
    """Draw overlapping radar plots for selected playlists."""
    angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    for name in playlists:
        row = df.filter(pl.col("playlist_name") == name)
        if row.is_empty():
            continue
        values = [float(row[f][0]) for f in features]
        values += values[:1]
        ax.plot(angles, values, label=name, linewidth=2)
        ax.fill(angles, values, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features, fontsize=9)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    ax.set_title("Audio Feature Profiles")
    plt.tight_layout()
    plt.show()

# Compare first 4 playlists (adjust as desired)
sample_playlists = playlist_audio["playlist_name"].to_list()[:4]
radar_plot(playlist_audio, sample_playlists, RADAR_FEATURES)

## 7. Genre Distribution

In [ ]:
# genre_cat / my_genre distribution across library
for col_name in ["genre_cat", "my_genre", "gen_4"]:
    if col_name in corpus.columns:
        counts = (
            corpus.group_by(col_name)
            .len()
            .sort("len", descending=True)
            .filter(pl.col(col_name).is_not_null())
        )
        fig, ax = plt.subplots(figsize=(10, max(4, len(counts) * 0.3)))
        ax.barh(counts[col_name].to_list()[::-1], counts["len"].to_list()[::-1])
        ax.set_xlabel("Track count")
        ax.set_title(f"{col_name} Distribution")
        plt.tight_layout()
        plt.show()

In [ ]:
# Genre hierarchy: gen_4 → gen_6 → gen_8 treemap (as table)
hierarchy_cols = ["gen_4", "gen_6", "gen_8"]
available_h = [c for c in hierarchy_cols if c in corpus.columns]

if len(available_h) >= 2:
    hierarchy = (
        corpus.filter(pl.col(available_h[0]).is_not_null())
        .group_by(available_h)
        .len()
        .sort(available_h + ["len"], descending=[False] * len(available_h) + [True])
    )
    print(f"Genre hierarchy ({' → '.join(available_h)}):")
    hierarchy.head(30)

In [ ]:
conn.close()